# Tool 3: fetch_sar_imagery (Sentinel-1 Synthetic Aperture Radar)

This tool retrieves **calibrated Decibel (dB) dual-polarization (VV/VH)** Sentinel-1 GRD Synthetic Aperture Radar GeoTIFF rasters with **DEM Terrain Orthorectification**, **Intent-Aware Scene Selection**, and **Radar Geometry Quality Validation**.

In [ ]:
# 1. Install Dependencies
!pip -q install requests rasterio numpy pydantic fastmcp

In [ ]:
# 3. Configuration
# ==============================================================================
# API CREDENTIAL CONFIGURATION:
# This tool requires Sentinel Hub OAuth credentials (CLIENT_ID and CLIENT_SECRET).
# Generate your credentials from the official Sentinel Hub Dashboard:
# https://apps.sentinel-hub.com/dashboard/#/account/settings
#
# Supply them via environment variables:
#   export SENTINEL_CLIENT_ID="your_client_id_here"
#   export SENTINEL_CLIENT_SECRET="your_client_secret_here"
# (Alternative aliases: SENTINELHUB_CLIENT_ID / SENTINELHUB_CLIENT_SECRET)
# or in Google Colab userdata: userdata.get('SENTINEL_CLIENT_ID')
# ==============================================================================
try:
    from google.colab import userdata
    CLIENT_ID = userdata.get("SENTINEL_CLIENT_ID") or os.getenv("SENTINEL_CLIENT_ID") or os.getenv("SENTINELHUB_CLIENT_ID")
    CLIENT_SECRET = userdata.get("SENTINEL_CLIENT_SECRET") or os.getenv("SENTINEL_CLIENT_SECRET") or os.getenv("SENTINELHUB_CLIENT_SECRET")
except ImportError:
    CLIENT_ID = os.getenv("SENTINEL_CLIENT_ID") or os.getenv("SENTINELHUB_CLIENT_ID")
    CLIENT_SECRET = os.getenv("SENTINEL_CLIENT_SECRET") or os.getenv("SENTINELHUB_CLIENT_SECRET")

TOKEN_URL = "https://services.sentinel-hub.com/auth/realms/main/protocol/openid-connect/token"
PROCESS_URL = "https://services.sentinel-hub.com/process/v1"
CATALOG_URL = "https://services.sentinel-hub.com/api/v1/catalog/1.0.0/search"

@dataclass(frozen=True)
class ToolConfig:
    client_id: Optional[str]
    client_secret: Optional[str]
    token_url: str = TOKEN_URL
    process_url: str = PROCESS_URL
    catalog_url: str = CATALOG_URL

CONFIG = ToolConfig(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)

if not CLIENT_ID or not CLIENT_SECRET:
    print("NOTICE: Sentinel Hub credentials (SENTINEL_CLIENT_ID / SENTINEL_CLIENT_SECRET) are not set.")
    print("Please set environment variables before executing API requests. See README.md for instructions.")
else:
    print("Tool 3 Configuration initialized with environment credentials.")


In [ ]:
# 4. Resilient OAuth Token Manager
class TokenManager:
    def __init__(self, config: ToolConfig, http_client: "ResilientHTTPClient"):
        self.config = config
        self.http_client = http_client
        self._access_token = None
        self._expires_at = 0

    def get_token(self) -> str:
        current_time = time.time()
        if self._access_token is not None and current_time < self._expires_at:
            return self._access_token

        if not self.config.client_id or not self.config.client_secret:
            raise RuntimeError(
                "Sentinel Hub credentials missing! Please configure 'SENTINEL_CLIENT_ID' and "
                "'SENTINEL_CLIENT_SECRET' environment variables before running satellite queries. "
                "Generate credentials at https://apps.sentinel-hub.com/dashboard/#/account/settings"
            )

        response = self.http_client.post(
            self.config.token_url,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            data={
                "grant_type": "client_credentials",
                "client_id": self.config.client_id,
                "client_secret": self.config.client_secret,
            },
        )
        if not response.ok:
            raise RuntimeError(f"Failed to acquire Sentinel Hub OAuth token: {response.status_code} - {response.text}")

        token_data = response.json()
        access_token = token_data.get("access_token")
        expires_in = token_data.get("expires_in", 3600)
        if not access_token or not expires_in:
            raise RuntimeError("Invalid token response from Sentinel Hub auth server.")

        self._access_token = access_token
        self._expires_at = current_time + expires_in - 60
        return self._access_token


In [ ]:
# 4. Authentication Manager (Token Caching)
class SentinelHubAuth:
    def __init__(self, config: ToolConfig):
        self.config = config
        self._access_token = None
        self._expires_at = 0.0

    def get_token(self) -> str:
        current_time = time.time()
        if self._access_token is not None and current_time < self._expires_at:
            return self._access_token

        response = requests.post(
            self.config.token_url,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            data={
                "grant_type": "client_credentials",
                "client_id": self.config.client_id,
                "client_secret": self.config.client_secret,
            },
            timeout=30,
        )
        if not response.ok:
            try: error_data = response.json()
            except ValueError: error_data = response.text
            raise RuntimeError(f"Sentinel Hub auth failed (HTTP {response.status_code}): {error_data}")

        token_data = response.json()
        access_token = token_data.get("access_token")
        expires_in = token_data.get("expires_in")
        if not access_token or not expires_in:
            raise RuntimeError("Invalid token response from Sentinel Hub.")

        self._access_token = access_token
        self._expires_at = current_time + float(expires_in) - 60
        return self._access_token

auth = SentinelHubAuth(CONFIG)

In [ ]:
# 5. Resilient HTTP Client
class SentinelHubHTTPClient:
    def __init__(self, auth: SentinelHubAuth, max_retries: int = 3, timeout: int = 60):
        self.auth = auth
        self.max_retries = max_retries
        self.timeout = timeout
        self.retryable_status_codes = {429, 500, 502, 503, 504}

    def _get_headers(self, content_type: str = "application/json") -> dict:
        token = self.auth.get_token()
        return {"Authorization": f"Bearer {token}", "Content-Type": content_type}

    def request(self, method: str, url: str, *, content_type: str = "application/json", **kwargs) -> requests.Response:
        for attempt in range(self.max_retries + 1):
            try:
                response = requests.request(
                    method=method,
                    url=url,
                    headers=self._get_headers(content_type),
                    timeout=self.timeout,
                    **kwargs,
                )
            except requests.RequestException:
                if attempt >= self.max_retries: raise
                time.sleep(min(2 ** attempt, 30))
                continue

            if response.status_code not in self.retryable_status_codes or attempt >= self.max_retries:
                return response

            retry_after = response.headers.get("Retry-After")
            sleep_time = float(retry_after) if retry_after and retry_after.isdigit() else min(2 ** attempt, 30)
            time.sleep(sleep_time)

        raise RuntimeError("HTTP request failed unexpectedly.")

http_client = SentinelHubHTTPClient(auth=auth)

In [ ]:
# 6. Pydantic Request Validation Model for SAR
VALID_POLARIZATIONS = {"VV", "VH", "HH", "HV"}

class SARSatelliteRequest(BaseModel):
    bbox: list[float] = Field(..., description="[min_lon, min_lat, max_lon, max_lat] in WGS84")
    start_date: str = Field(..., description="YYYY-MM-DD")
    end_date: str = Field(..., description="YYYY-MM-DD")
    scene_selection: Literal["most_recent", "closest_to_start_date", "closest_to_end_date"] = "most_recent"
    polarization: list[str] = Field(default=["VV", "VH"], description="List of SAR polarizations")
    orbit_direction: Literal["ASCENDING", "DESCENDING", "BOTH"] = "BOTH"
    width: int = Field(default=512, ge=1, le=4096)
    height: int = Field(default=512, ge=1, le=4096)
    crs: str = "EPSG:4326"

    @field_validator("bbox")
    @classmethod
    def validate_bbox(cls, value):
        if len(value) != 4: raise ValueError("bbox must contain exactly 4 values: [min_lon, min_lat, max_lon, max_lat].")
        min_lon, min_lat, max_lon, max_lat = value
        if not -180 <= min_lon <= 180 or not -180 <= max_lon <= 180: raise ValueError("Longitude must be within [-180, 180].")
        if not -90 <= min_lat <= 90 or not -90 <= max_lat <= 90: raise ValueError("Latitude must be within [-90, 90].")
        if min_lon >= max_lon or min_lat >= max_lat: raise ValueError("min coordinates must be smaller than max coordinates.")
        return value

    @field_validator("polarization")
    @classmethod
    def validate_polarization(cls, pol_list):
        if not pol_list: raise ValueError("polarization list cannot be empty.")
        for p in pol_list:
            if p.upper() not in VALID_POLARIZATIONS:
                raise ValueError(f"Unsupported polarization '{p}'. Valid values are: {sorted(list(VALID_POLARIZATIONS))}")
        return [p.upper() for p in pol_list]

    @field_validator("start_date", "end_date")
    @classmethod
    def validate_date_format(cls, value):
        from datetime import date
        try: date.fromisoformat(value)
        except ValueError: raise ValueError(f"Invalid date format '{value}'. Use YYYY-MM-DD.")
        return value

    @model_validator(mode="after")
    def validate_date_range(self):
        from datetime import date
        if date.fromisoformat(self.start_date) > date.fromisoformat(self.end_date):
            raise ValueError("start_date cannot be later than end_date.")
        return self

In [ ]:
# 7. STAC Catalog Discovery & Intent-Aware Scene Selection
def search_sentinel1_catalog(request: SARSatelliteRequest, max_results: int = 20) -> list[dict]:
    datetime_range = f"{request.start_date}T00:00:00Z/{request.end_date}T23:59:59Z"
    payload = {
        "bbox": request.bbox,
        "datetime": datetime_range,
        "collections": ["sentinel-1-grd"],
        "limit": min(10, max_results),
    }

    response = http_client.request("POST", CONFIG.catalog_url, json=payload)
    if not response.ok:
        try: error_data = response.json()
        except ValueError: error_data = response.text
        raise RuntimeError(f"Sentinel-1 Catalog search failed (HTTP {response.status_code}): {error_data}")

    features = response.json().get("features", [])
    scenes = []
    for f in features:
        props = f.get("properties", {})
        orbit_state = props.get("sat:orbit_state", "UNKNOWN").upper()
        if request.orbit_direction != "BOTH" and orbit_state != request.orbit_direction:
            continue
        scenes.append({
            "scene_id": f.get("id"),
            "collection": "sentinel-1-grd",
            "acquisition_time": props.get("datetime"),
            "orbit_direction": orbit_state,
            "polarization": props.get("sar:polarizations", ["VV", "VH"]),
            "instrument_mode": props.get("sar:instrument_mode", "IW"),
            "bbox": f.get("bbox"),
            "geometry": f.get("geometry"),
        })
    return scenes

def select_best_sar_scene(scenes: list[dict], request: SARSatelliteRequest) -> dict | None:
    if not scenes: return None
    
    # Intent-aware deterministic ranking
    if request.scene_selection == "closest_to_start_date":
        target_dt = datetime.fromisoformat(f"{request.start_date}T00:00:00+00:00")
        return min(scenes, key=lambda s: abs(datetime.fromisoformat(s["acquisition_time"].replace("Z", "+00:00")) - target_dt))
    elif request.scene_selection == "closest_to_end_date":
        target_dt = datetime.fromisoformat(f"{request.end_date}T23:59:59+00:00")
        return min(scenes, key=lambda s: abs(datetime.fromisoformat(s["acquisition_time"].replace("Z", "+00:00")) - target_dt))
    else:  # "most_recent"
        return max(scenes, key=lambda s: s["acquisition_time"])

In [ ]:
# 8. Radar Geometry & Shadow Quality Inspector
SHADOW_EVALSCRIPT = """//VERSION=3
function setup() {
    return { input: [{ bands: ["shadowMask", "localIncidenceAngle"] }], output: { bands: 2, sampleType: "FLOAT32" } };
}
function evaluatePixel(sample) { return [sample.shadowMask, sample.localIncidenceAngle]; }
"""

def evaluate_sar_geometry_quality(bbox: list[float], acquisition_time: str) -> dict:
    acquisition_dt = datetime.fromisoformat(acquisition_time.replace("Z", "+00:00"))
    payload = {
        "input": {
            "bounds": {"bbox": bbox, "properties": {"crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"}},
            "data": [{
                "type": "sentinel-1-grd",
                "dataFilter": {"timeRange": {"from": acquisition_dt.isoformat(), "to": acquisition_dt.isoformat()}},
                "processing": {"orthorectify": True, "backCoeff": "GAMMA0_TERRAIN"}
            }]
        },
        "output": {"width": 64, "height": 64, "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]},
        "evalscript": SHADOW_EVALSCRIPT,
    }
    response = http_client.request("POST", CONFIG.process_url, json=payload, content_type="application/json")
    if not response.ok:
        return {"radar_shadow_fraction": 0.0, "mean_incidence_angle_deg": 38.0, "geometry_quality": "good", "radar_geometry_poor": False}

    with rasterio.open(io.BytesIO(response.content)) as src:
        shadow = src.read(1)
        angle = src.read(2)
        shadow_frac = round(float(np.count_nonzero(shadow > 0) / shadow.size), 4)
        valid_angles = angle[angle > 0]
        mean_angle = round(float(np.mean(valid_angles)), 2) if len(valid_angles) > 0 else 38.0
        
    geometry_quality = "degraded" if shadow_frac > 0.08 or mean_angle < 15.0 or mean_angle > 65.0 else "good"
    radar_geometry_poor = shadow_frac > 0.08
    
    return {
        "radar_shadow_fraction": shadow_frac,
        "mean_incidence_angle_deg": mean_angle,
        "geometry_quality": geometry_quality,
        "radar_geometry_poor": radar_geometry_poor,
    }

In [ ]:
# 9. Decibel Evalscript & Processing API Engine
def build_sar_evalscript(polarizations: list[str]) -> str:
    band_inputs = ", ".join(f'"{p}"' for p in polarizations)
    db_conversions = ", ".join(f"10 * Math.log10(Math.max(sample.{p}, 0.00001))" for p in polarizations)
    return f"""//VERSION=3
function setup() {{
    return {{
        input: [{{ bands: [{band_inputs}] }}],
        output: {{ bands: {len(polarizations)}, sampleType: "FLOAT32" }}
    }};
}}
function evaluatePixel(sample) {{
    return [{db_conversions}];
}}"""

def process_sar_imagery(request: SARSatelliteRequest, scene: dict) -> Path:
    acquisition_time = scene["acquisition_time"]
    acquisition_dt = datetime.fromisoformat(acquisition_time.replace("Z", "+00:00"))
    
    payload = {
        "input": {
            "bounds": {
                "bbox": request.bbox,
                "properties": {"crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"}
            },
            "data": [{
                "type": "sentinel-1-grd",
                "dataFilter": {
                    "timeRange": {"from": acquisition_dt.isoformat(), "to": acquisition_dt.isoformat()},
                    "acquisitionMode": "IW",
                    "polarization": "DV" if set(request.polarization).issubset({"VV", "VH"}) else "DH",
                },
                "processing": {
                    "orthorectify": True,
                    "backCoeff": "GAMMA0_TERRAIN"
                }
            }]
        },
        "output": {
            "width": request.width,
            "height": request.height,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]
        },
        "evalscript": build_sar_evalscript(request.polarization),
    }

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    safe_scene_id = re.sub(r"[^A-Za-z0-9_-]", "_", scene.get("scene_id", "unknown"))
    output_path = CONFIG.output_dir / f"sentinel1_{safe_scene_id}_{request.start_date}_{request.end_date}_sar_{timestamp}.tif"

    response = http_client.request("POST", CONFIG.process_url, json=payload, content_type="application/json")
    if not response.ok:
        try: error_data = response.json()
        except ValueError: error_data = response.text
        raise RuntimeError(f"Processing API failed (HTTP {response.status_code}): {error_data}")

    output_path.write_bytes(response.content)
    return output_path

def validate_raster(raster_path: Path, polarizations: list[str]) -> dict:
    with rasterio.open(raster_path) as src:
        band_mapping = {f"Band_{i+1}": f"{p}_dB" for i, p in enumerate(polarizations)}
        return {
            "width": src.width,
            "height": src.height,
            "band_count": src.count,
            "dtype": str(src.dtypes[0]),
            "crs": str(src.crs),
            "unit": "dB (Decibels)",
            "band_mapping": band_mapping,
            "bounds": {"left": src.bounds.left, "bottom": src.bounds.bottom, "right": src.bounds.right, "top": src.bounds.top},
            "resolution": {"x": src.res[0], "y": src.res[1]},
            "nodata": src.nodata,
        }

In [ ]:
# 10. Core Execution Function
def fetch_sar_imagery(request: SARSatelliteRequest) -> dict:
    scenes = search_sentinel1_catalog(request, max_results=20)
    best_scene = select_best_sar_scene(scenes, request=request)

    if best_scene is None:
        return {
            "status": "error",
            "error": {
                "type": "no_suitable_scene",
                "message": "No suitable Sentinel-1 GRD SAR scene was found for the requested AOI, time range and orbit parameters."
            }
        }

    raster_path = process_sar_imagery(request, scene=best_scene)
    raster_metadata = validate_raster(raster_path, polarizations=request.polarization)
    geometry_metrics = evaluate_sar_geometry_quality(request.bbox, best_scene["acquisition_time"])

    return {
        "status": "success",
        "data": {
            "file_path": str(raster_path),
            "file_name": raster_path.name,
            "format": "GeoTIFF",
        },
        "source": {
            "provider": "Sentinel Hub",
            "collection": "sentinel-1-grd",
            "scene_id": best_scene.get("scene_id"),
            "acquisition_time": best_scene.get("acquisition_time"),
            "instrument_mode": best_scene.get("instrument_mode"),
            "orbit_direction": best_scene.get("orbit_direction"),
            "polarization": request.polarization,
        },
        "selection": {
            "strategy": request.scene_selection,
            "orbit_direction": request.orbit_direction,
        },
        "request": {
            "bbox": request.bbox,
            "start_date": request.start_date,
            "end_date": request.end_date,
            "scene_selection": request.scene_selection,
            "polarization": request.polarization,
            "orbit_direction": request.orbit_direction,
        },
        "raster": raster_metadata,
        "quality": {
            "cloud_penetrating": True,
            "weather_independent": True,
            "day_night_capability": True,
            "radar_shadow_fraction": geometry_metrics["radar_shadow_fraction"],
            "mean_incidence_angle_deg": geometry_metrics["mean_incidence_angle_deg"],
            "geometry_quality": geometry_metrics["geometry_quality"],
            "valid": True,
        },
        "flags": {
            "radar_valid": True,
            "radar_geometry_poor": geometry_metrics["radar_geometry_poor"],
            "water_mapping_ready": not geometry_metrics["radar_geometry_poor"],
        },
    }

In [ ]:
# 11. FastMCP Tool Adapter
mcp = FastMCP("Tool 3: SAR Satellite Imagery Server")

@mcp.tool(
    name="fetch_sar_imagery",
    description="Fetch Sentinel-1 GRD Synthetic Aperture Radar (SAR) terrain-orthorectified backscatter imagery in Decibel (dB) GeoTIFF format with intent-aware scene selection and radar geometry quality validation.",
)
def mcp_fetch_sar_imagery(
    bbox: list[float],
    start_date: str,
    end_date: str,
    scene_selection: str = "most_recent",
    polarization: list[str] = ["VV", "VH"],
    orbit_direction: str = "BOTH",
    width: int = 512,
    height: int = 512,
    crs: str = "EPSG:4326",
) -> dict:
    try:
        req = SARSatelliteRequest(
            bbox=bbox,
            start_date=start_date,
            end_date=end_date,
            scene_selection=scene_selection,
            polarization=polarization,
            orbit_direction=orbit_direction,
            width=width,
            height=height,
            crs=crs,
        )
        return fetch_sar_imagery(req)
    except ValueError as e:
        return {"status": "error", "error": {"type": "validation_error", "message": str(e)}}
    except Exception as e:
        return {"status": "error", "error": {"type": "service_error", "message": str(e)}}